# 02 - Header-Based Document Splitting

## Purpose

This notebook applies a structured text-splitting approach to the extracted PDF transcript.

The goal is to check whether the uploaded PDF text contains recognizable section headers. If headers are found, the splitter preserves them as metadata so later retrieval can include richer source context.

This is different from the first implementation, which used only character-based splitting.

## Main Changes in Version 2

This version adds:

- Header-aware document splitting
- Metadata preservation for sections
- Cleaner preparation before token-based splitting
- A more structured document list for retrieval

## Input

This notebook uses the extracted transcript from Notebook 01:

- `string_list_concat`

## Main Output

- `docs_list_md_split`

## Notebook Flow

```text
Full extracted PDF transcript
        ↓
Define header rules
        ↓
Apply MarkdownHeaderTextSplitter
        ↓
Create structured LangChain documents
        ↓
Inspect metadata and content
        ↓
Prepare documents for token splitting
´´´

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/Volumes/workspace/365pdf/365pdfupload/Introduction_to_Tableau.pdf"

loader_pdf = PyPDFLoader(pdf_path)
docs_list = loader_pdf.load()

string_list_concat = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    string_list_concat += f"\n\n--- Page {page_number} ---\n"
    string_list_concat += page_text

print("PDF reloaded successfully")
print("Number of pages:", len(docs_list))
print("Transcript characters:", len(string_list_concat))
print(string_list_concat[:1000])

In [0]:
markdown_transcript = ""

for doc in docs_list:
    page_number = doc.metadata.get("page", "unknown")
    page_text = doc.page_content

    markdown_transcript += f"\n\n# Introduction to Tableau\n"
    markdown_transcript += f"## Page {page_number}\n\n"
    markdown_transcript += page_text

print("Markdown-style transcript created")
print("Characters:", len(markdown_transcript))
print(markdown_transcript[:1200])

In [0]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Section Title"),
    ("##", "Page Title")
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs_list_md_split = md_splitter.split_text(markdown_transcript)

print("Header-based documents created:", len(docs_list_md_split))

In [0]:
for i, doc in enumerate(docs_list_md_split[:5]):
    print(f"\nDocument {i}")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:700])
    print("-" * 80)

In [0]:
print("Final object ready for next notebook:")
print("docs_list_md_split")
print("Number of documents:", len(docs_list_md_split))